# 05: model v2

Rule: every choice comes from 5-fold CV on the training set. The test set is opened once, at the very end.
Goal: precision >= 0.80 with the highest recall.

In [1]:
import os
# One math thread per worker: parallel CV overloads the CPU otherwise (same fix as notebook 04).
for var in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1"

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

sys.path.insert(0, str(Path("..").resolve()))   # so "from src..." works from notebooks/
from src.thresholds import pick_threshold

DATA_PATH = Path("..") / "data" / "fake_job_postings.csv"
MODEL_DIR = Path("..") / "models"
RESULTS_DIR = Path("..") / "results"
RANDOM_STATE = 42
MIN_PRECISION = 0.80   # my goal, 2026-10-31

In [2]:
df = pd.read_csv(DATA_PATH)
TEXT_COLUMNS = ["title", "company_profile", "description", "requirements", "benefits"]
df["text"] = df[TEXT_COLUMNS].fillna("").agg(" ".join, axis=1)
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
df["salary_listed"] = df["salary_range"].notna().astype(int)
df["has_company_profile"] = df["company_profile"].notna().astype(int)
METADATA_COLUMNS = ["has_company_logo", "has_questions", "telecommuting", "salary_listed", "has_company_profile"]

X = df[["text"] + METADATA_COLUMNS]
y = df["fraudulent"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train {len(y_train):,} ({y_train.mean():.2%} fake), Test {len(y_test):,}")
print("Expected (notebook 03): Train 12,852 (4.47%), Test 3,214")
# 🔒 X_test / y_test are not used again until the final section.

Train 12,852 (4.47% fake), Test 3,214
Expected (notebook 03): Train 12,852 (4.47%), Test 3,214


Caveat: v1's settings (C=10, word 1-2 grams) were picked by a grid search on these same 5 folds, so its
out-of-fold numbers are slightly optimistic. v2 is tuned on the same folds, so the comparison stays fair.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
v1 = clone(joblib.load(MODEL_DIR / "job_trust_v1.joblib")["pipeline"])   # same settings, untrained
v1_oof = cross_val_predict(v1, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
print(f"v1 pooled out-of-fold PR-AUC: {average_precision_score(y_train, v1_oof):.3f} "
      "(nb 03 per-fold mean 0.879; pooled differs slightly)")

v1 pooled out-of-fold PR-AUC: 0.878 (nb 03 per-fold mean 0.879; pooled differs slightly)


In [4]:
v1_t = pick_threshold(v1_oof, y_train, min_precision=MIN_PRECISION)
v1_pred = (v1_oof >= v1_t).astype(int)
v1_row = pd.DataFrame([{
    "model": "v1", "threshold": v1_t,
    "precision": precision_score(y_train, v1_pred), "recall": recall_score(y_train, v1_pred),
    "flagged": int(v1_pred.sum()), "false_alarms": int(((v1_pred == 1) & (y_train == 0)).sum()),
}])
RESULTS_DIR.mkdir(exist_ok=True)
v1_row.to_csv(RESULTS_DIR / "v1_cv_at_p80.csv", index=False)
v1_row.round(3)

,model,threshold,precision,recall,flagged,false_alarms
0,v1,0.618,0.8,0.787,565,113


In [5]:
false_alarm = (v1_pred == 1) & (y_train == 0).to_numpy()
fa = df.loc[X_train.index[false_alarm]].assign(v1_prob=v1_oof[false_alarm])
fa = fa.sort_values("v1_prob", ascending=False).head(25)
fa[["v1_prob", "job_id", "title", "location", "salary_range"] + TEXT_COLUMNS[1:]].to_csv(
    Path("..") / "data" / "v1_false_alarms_cv.csv", index=False)
print(f"{false_alarm.sum()} false alarms in total; top 25 saved to data/v1_false_alarms_cv.csv")

113 false alarms in total; top 25 saved to data/v1_false_alarms_cv.csv
